# 12 · T1_Speech vs T2_Text on the same ids

How much does going through speech cost, and does the loss follow WER?
Outputs for the midterm slides: headline table, BERTScore by WER bin (+ plot), 4–6 examples.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root: common.py, speech.py

from common import *
from speech import *

import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.max_colwidth", 120)

In [ ]:
SPLIT = "dev"
ASR_MODEL = "whisper-large-v3-turbo"
VQA_MODEL, PROMPT_ID = "qwen25vl7b", "p2"     # <- same as in 11_cascade

ASR_FILE = asr_path(SPLIT, ASR_MODEL)
SPEECH = preds_path("t1_speech_cascade", SPLIT, f"whisperturbo+{VQA_MODEL}", PROMPT_ID)
TEXT = preds_path("t2_text", SPLIT, VQA_MODEL, PROMPT_ID)

df = join_t1_t2(ASR_FILE, SPEECH, TEXT, SPLIT)
print(len(df), "common ids")

## 1. Headline (slide table)

In [ ]:
head = {
    "split": SPLIT, "n": len(df), "vqa": VQA_MODEL, "prompt_id": PROMPT_ID, "asr": ASR_MODEL,
    "wer_corpus": round(wer(df["transcript"].tolist(), df["question"].tolist()), 4),
    "f1_t2_text": round(df["f1_text"].mean(), 4),
    "f1_t1_speech_cascade": round(df["f1_speech"].mean(), 4),
    "delta": round(df["delta"].mean(), 4),
    "share_speech_worse_by_0.02": round((df["delta"] < -0.02).mean(), 4),
}
with open(RESULTS_DIR / f"t1_vs_t2_{SPLIT}_{VQA_MODEL}_{PROMPT_ID}.json", "w") as f:
    json.dump(head, f, indent=2)
pd.Series(head)

## 2. BERTScore by WER bin

In [ ]:
bins = wer_bin_table(df)
bins.to_json(RESULTS_DIR / f"t1_vs_t2_wer_bins_{SPLIT}_{VQA_MODEL}_{PROMPT_ID}.json", indent=2)
bins

In [ ]:
x = range(len(bins))
fig, ax = plt.subplots(figsize=(7, 3.5))
ax.bar([i - 0.2 for i in x], bins["f1_text"], width=0.4, label="T2_Text (text question)")
ax.bar([i + 0.2 for i in x], bins["f1_speech"], width=0.4, label="T1_Speech (cascade)")
ax.set_xticks(list(x), [f"{b}\nn={n}" for b, n in zip(bins.index, bins["n"])])
lo = min(bins["f1_text"].min(), bins["f1_speech"].min())
ax.set_ylim(lo - 0.03, None)
ax.set_xlabel("WER of the transcript"); ax.set_ylabel("BERTScore F1")
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(RESULTS_DIR / f"t1_vs_t2_wer_bins_{SPLIT}_{VQA_MODEL}_{PROMPT_ID}.png", dpi=200)

In [ ]:
# rank correlation: does a worse transcript mean a bigger drop?
print("Spearman(WER, delta) =", round(df["wer"].corr(df["delta"], method="spearman"), 3))

## 3. Examples for the slides

In [ ]:
cols = ["wer", "delta", "question", "transcript", "answer", "pred_text", "pred_speech"]
print("ASR error -> answer broke")
df[df["wer"] > 0].sort_values("delta").head(5)[cols]

In [ ]:
print("ASR error, answer survived")
df[(df["wer"] > 0.1) & (df["delta"] > -0.01)].sort_values("wer", ascending=False).head(5)[cols]

In [ ]:
df["missed"] = [missed_names(q, t) for q, t in zip(df["question"], df["transcript"])]
m = df["missed"].map(len) > 0
print(f"lost a name/place/dish: {m.mean():.1%} of questions; "
      f"mean delta {df.loc[m, 'delta'].mean():.4f} vs {df.loc[~m, 'delta'].mean():.4f} without")
df[m].sort_values("delta").head(5)[["missed"] + cols]

## 4. By category
The released EN data (train, dev, devtest) has no `category` / `country`, and devtest has no `answer`.
Kept for the test-phase gold or a future labelled release; prints a note otherwise.

In [ ]:
if df["category"].notna().any():
    display(df.groupby("category")[["f1_text", "f1_speech", "delta", "wer"]].mean().round(4)
              .sort_values("delta"))
else:
    print("no category in", SPLIT)